# 1. Fourier fitting and explicit validation

Only this notebook generates Fourier coefficients. It saves immutable fits; no classical dynamics are triggered until you explicitly populate `selected_for_dynamics`. Raw fit screening and accepted reduced fits are shown **separately**, so rejected candidates do not introduce misleading NaNs into the saved-fit table.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Locate the checkout so notebooks work without an editable install.
# Supported working directories: repository root or a directory inside it.
_here = Path.cwd().resolve()
ROOT = next(
    (directory for directory in (_here, *_here.parents)
     if (directory / "src" / "bosonic_algo" / "__init__.py").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError(
        "Cannot locate the bosonic_algo checkout from the notebook kernel's "
        "working directory. Start Jupyter in the repository root or its "
        "notebooks/ folder (or change the kernel working directory). "
        "For another environment, install from the repository root with "
        "'python -m pip install -e .'."
    )
_src = str(ROOT / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from bosonic_algo import generate_fits, read_yaml, selected_fit_classical_validation, list_fits
from bosonic_algo import fourier as _fourier
if getattr(_fourier, "SUMMARY_SCHEMA_VERSION", 0) < 2:
    raise RuntimeError(
        "Outdated Fourier workflow detected in the Jupyter kernel. "
        "Run 'git pull' on the refactor branch and restart the notebook kernel. "
        "The old version mixes rejected candidates with saved fits, causing NaNs."
    )
print("Using Fourier fitting module:", Path(_fourier.__file__).resolve())
FIT_ROOT = ROOT / "data" / "fits"
RESULT_ROOT = ROOT / "results"
settings = read_yaml(ROOT / "configs" / "fourier.yaml")

## Generate reduced fits (no time evolution)

`rows` contains **only saved reduced Fourier fits**, with uniform columns. All raw candidates, including rejected ones, are written to `data/fits/raw_fit_summary.csv`.

In [ ]:
rows = generate_fits(settings, root=FIT_ROOT)
saved_df = pd.DataFrame(rows)
print(f"Saved reduced fits: {len(rows)}")
if saved_df.empty:
    print("No raw candidates passed screening; inspect the raw summary below.")
else:
    threshold = settings.get("raw_v_rmse_max", 0.01)
    above = saved_df[saved_df["v_rmse"] > threshold]
    if len(above):
        print(f"Note: {len(above)} reduced fits exceed raw-screen threshold "
              f"({threshold:g}) AFTER selecting sparse modes. Review their v_rmse and forces.")
    if saved_df.isna().any().any():
        raise RuntimeError("Saved-fit table contains missing values; verify imported module version.")
saved_df

## Raw Fourier-fit screening

The raw table contains **every** Fourier candidate and its raw potential/force metrics. A `rejected` status means the raw potential RMSE exceeds `raw_v_rmse_max`; it was not reduced or propagated. This is expected, not missing numerical data.

In [ ]:
raw_df = pd.read_csv(FIT_ROOT / "raw_fit_summary.csv")
print("Raw candidates:", len(raw_df), "| Passed:", (raw_df["status"] == "passed").sum(),
      "| Rejected:", (raw_df["status"] == "rejected").sum())
print("Counts by box_r and status:")
print(raw_df.groupby(["fourier_box_r", "status"]).size().unstack(fill_value=0))
if raw_df.select_dtypes(include="number").isna().any().any():
    raise RuntimeError("Non-finite numeric metrics in raw-fit summary; inspect the solver.")
raw_df

## Choose fit IDs from this sweep

The table below contains only saved fits from the current sweep, **not historical metadata**. The old `list_fits()` directory scan can mix runs created with different reporting versions and display missing metrics as NaN. Use `list_fits(FIT_ROOT)` separately when you intentionally want to browse previous experiments. Only explicitly selected IDs are propagated.

In [ ]:
if saved_df.empty:
    print("No saved fits in this sweep. Inspect the raw-screening table.")
else:
    columns = ["fit_id", "fourier_box_R", "fourier_box_r", "n_modes",
               "raw_rmse", "v_rmse", "force_combined_rmse", "path"]
    display(saved_df[columns].sort_values("v_rmse"))

In [ ]:
selected = settings.get("selected_for_dynamics", [])
if not selected:
    print("No classical dynamics requested. Set selected_for_dynamics explicitly.")
else:
    validated = selected_fit_classical_validation(selected, fit_root=FIT_ROOT, result_root=RESULT_ROOT)
    for run in validated:
        print(run["fit_id"], "exact cached:", run["exact"][2], "fitted cached:", run["fitted"][2])